In [ ]:
import os
root = '/content/drive/MyDrive/J26-DS-320/datasets/gmdcsa24'
for dirpath, dirs, files in os.walk(root):
    files = [f for f in files if not f.startswith('.')]
    print(dirpath.replace(root, '.'), '| subfolders:', dirs, '| files:', len(files))
    for f in files[:6]:
        print('    ', f)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
base = '/content/drive/MyDrive'
print("MyDrive exists:", os.path.exists(base))
if os.path.exists(base):
    print([d for d in os.listdir(base) if not d.startswith('.')][:40])
    p = os.path.join(base, 'J26-DS-320', 'datasets')
    print("datasets exists:", os.path.exists(p))
    if os.path.exists(p):
        print(os.listdir(p))

In [ ]:
import os, pandas as pd
root = '/content/drive/MyDrive/J26-DS-320/datasets/gmdcsa24'
print("top level:", [d for d in os.listdir(root) if not d.startswith('.')])

# 1. how many videos per subject / category
for s in sorted(os.listdir(root)):
    sp = os.path.join(root, s)
    if not os.path.isdir(sp) or s.startswith('.'): continue
    for cat in ['ADL', 'Fall']:
        cp = os.path.join(sp, cat)
        vids = [f for f in os.listdir(cp) if not f.startswith('.')] if os.path.isdir(cp) else []
        print(s, cat, len(vids), vids[:2])

# 2. peek at the label CSVs
for f in ['Fall.csv', 'ADL.csv']:
    df = pd.read_csv(os.path.join(root, 'Subject 1', f))
    print('\n', f, df.shape)
    print(df.columns.tolist())
    print(df.head(3).to_string())

In [ ]:
!pip -q install mediapipe
!wget -q -O pose_landmarker_lite.task https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task

import os, re, cv2, pickle, time
import numpy as np, pandas as pd
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

detector = vision.PoseLandmarker.create_from_options(vision.PoseLandmarkerOptions(
    base_options=python.BaseOptions(model_asset_path='pose_landmarker_lite.task'),
    output_segmentation_masks=False))

ROOT = '/content/drive/MyDrive/J26-DS-320/datasets/gmdcsa24'
OUT  = '/content/drive/MyDrive/J26-DS-320/datasets/gmdcsa24_features.pkl'

def frame_features(bgr):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    res = detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
    if not res.pose_landmarks: return None
    lm = res.pose_landmarks[0]
    xs, ys = [l.x for l in lm], [l.y for l in lm]
    w, h = max(xs)-min(xs), max(ys)-min(ys)
    return {"nose_y": lm[0].y,
            "shoulder_y": (lm[11].y + lm[12].y)/2,
            "hip_y": (lm[23].y + lm[24].y)/2,
            "aspect_ratio": w/h if h > 0.01 else 0,
            "width": w, "height": h}

In [ ]:
STRIDE = 2   # use every 2nd frame to halve the compute

# load previous progress if any
results = pickle.load(open(OUT, 'rb')) if os.path.exists(OUT) else {}

def parse_fall_interval(classes):
    m = re.search(r'Falling[^\[]*\[\s*([\d.]+)\s*to\s*([\d.]+)\s*\]', str(classes))
    return (float(m.group(1)), float(m.group(2))) if m else None

t0 = time.time()
for subj in sorted(d for d in os.listdir(ROOT) if d.startswith('Subject')):
    for cat, label in [('ADL', 0), ('Fall', 1)]:
        meta = pd.read_csv(f'{ROOT}/{subj}/{cat}.csv')
        meta.columns = [c.strip() for c in meta.columns]
        meta = meta.set_index('File Name')
        folder = f'{ROOT}/{subj}/{cat}'
        for fn in sorted(f for f in os.listdir(folder) if f.endswith('.mp4') and not f.startswith('.')):
            key = f'{subj}/{cat}/{fn}'
            if key in results: continue
            cap = cv2.VideoCapture(f'{folder}/{fn}')
            fps = cap.get(cv2.CAP_PROP_FPS) or 30
            feats, i = [], 0
            while True:
                ok, frame = cap.read()
                if not ok: break
                if i % STRIDE == 0:
                    f = frame_features(frame)
                    if f: f['t'] = i / fps
                    feats.append(f)
                i += 1
            cap.release()
            classes = meta.loc[fn, 'Classes'] if fn in meta.index else None
            results[key] = {"subject": subj, "label": label, "fps": fps, "stride": STRIDE,
                            "features": feats, "classes": classes,
                            "fall_interval": parse_fall_interval(classes) if label else None}
        pickle.dump(results, open(OUT, 'wb'))
        print(f'{subj} {cat} done | clips so far: {len(results)} | {time.time()-t0:.0f}s')

In [ ]:
n_none = sum(1 for r in results.values() for f in r['features'] if f is None)
n_all  = sum(len(r['features']) for r in results.values())
print(f'{len(results)} clips | frames with no person: {n_none}/{n_all} ({100*n_none/n_all:.1f}%)')
print('fps values:', sorted({round(r["fps"],1) for r in results.values()}))
print('falls missing interval:', [k for k,r in results.items() if r['label']==1 and r['fall_interval'] is None])

In [ ]:
# 1. why did 3 intervals fail to parse?
for k in ['Subject 3/Fall/01.mp4', 'Subject 3/Fall/02.mp4', 'Subject 4/Fall/09.mp4']:
    print(k, '->', repr(results[k]['classes']))

# 2. resample every clip to 100 timesteps by TIME (handles 15fps vs 30fps)
FEATURE_NAMES = ["nose_y", "shoulder_y", "hip_y", "aspect_ratio", "width", "height"]
T = 100

def clip_to_array(feats, T=T):
    valid = [f for f in feats if f is not None]
    if len(valid) < 5: return None
    t = np.array([f['t'] for f in valid])
    grid = np.linspace(t.min(), t.max(), T)
    return np.stack([np.interp(grid, t, [f[n] for f in valid]) for n in FEATURE_NAMES], axis=1).astype(np.float32)

X, y, subj, names, skipped = [], [], [], [], []
for k, r in results.items():
    arr = clip_to_array(r['features'])
    if arr is None: skipped.append(k); continue
    X.append(arr); y.append(r['label']); subj.append(r['subject']); names.append(k)

X, y, subj = np.array(X), np.array(y), np.array(subj)
print('X', X.shape, '| falls', (y==1).sum(), '| ADL', (y==0).sum(), '| skipped', skipped)
for s in sorted(set(subj)):
    m = subj == s
    print(s, 'clips', m.sum(), '| falls', (y[m]==1).sum())

pickle.dump({'X': X, 'y': y, 'subj': subj, 'names': names},
            open('/content/drive/MyDrive/J26-DS-320/datasets/gmdcsa24_arrays.pkl', 'wb'))

In [ ]:
import torch, torch.nn as nn, torch.optim as optim, copy, json
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class FallCNNLSTM(nn.Module):
    def __init__(self, num_features=6, cnn_channels=[32, 64, 128], lstm_hidden=128, lstm_layers=2):
        super().__init__()
        self.conv1 = nn.Conv1d(num_features, cnn_channels[0], kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(cnn_channels[0], cnn_channels[1], kernel_size=3, padding=1)
        self.conv3 = nn.Conv1d(cnn_channels[1], cnn_channels[2], kernel_size=3, padding=1)
        self.relu = nn.ReLU(); self.dropout = nn.Dropout(0.3)
        self.lstm = nn.LSTM(cnn_channels[2], lstm_hidden, num_layers=lstm_layers, batch_first=True, dropout=0.3)
        self.fc1 = nn.Linear(lstm_hidden, 64); self.fc2 = nn.Linear(64, 2)
    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.dropout(self.relu(self.conv1(x)))
        x = self.dropout(self.relu(self.conv2(x)))
        x = self.dropout(self.relu(self.conv3(x)))
        _, (h_n, _) = self.lstm(x.permute(0, 2, 1))
        return self.fc2(self.relu(self.fc1(h_n[-1])))

def loader(Xa, ya, shuffle):
    return DataLoader(TensorDataset(torch.tensor(Xa), torch.tensor(ya, dtype=torch.long)), batch_size=8, shuffle=shuffle)

def predict(model, Xa):
    model.eval()
    with torch.no_grad():
        return torch.argmax(model(torch.tensor(Xa).to(device)), dim=1).cpu().numpy()

def train_model(Xtr, ytr, Xva, yva, epochs=50, patience=10):
    model = FallCNNLSTM().to(device)
    w = torch.tensor([1.0 / c for c in np.bincount(ytr)], dtype=torch.float32).to(device)
    crit = nn.CrossEntropyLoss(weight=w)
    opt = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)
    best_f1, best_state, bad = -1, None, 0
    for ep in range(epochs):
        model.train()
        for bx, by in loader(Xtr, ytr, True):
            bx, by = bx.to(device), by.to(device)
            opt.zero_grad(); crit(model(bx), by).backward(); opt.step()
        f1 = f1_score(yva, predict(model, Xva), zero_division=0)
        if f1 > best_f1: best_f1, best_state, bad = f1, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state)
    return model

def metrics(yt, yp):
    tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
    return {"precision": float(precision_score(yt, yp, zero_division=0)),
            "recall": float(recall_score(yt, yp, zero_division=0)),
            "f1": float(f1_score(yt, yp, zero_division=0)),
            "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn)}

# ---------- Experiment A: leave-one-subject-out ----------
torch.manual_seed(42); np.random.seed(42)
loso, all_true, all_pred = {}, [], []
for held in sorted(set(subj)):
    te = subj == held
    Xtr, ytr = X[~te], y[~te]
    Xtr, Xva, ytr, yva = train_test_split(Xtr, ytr, test_size=0.15, stratify=ytr, random_state=42)
    m = train_model(Xtr, ytr, Xva, yva)
    yp = predict(m, X[te])
    loso[held] = metrics(y[te], yp)
    all_true += list(y[te]); all_pred += list(yp)
    print(held, loso[held])
pooled = metrics(np.array(all_true), np.array(all_pred))
print('\nPOOLED LOSO:', pooled)

# ---------- Experiment B: URFD-trained model tested on ALL GMDCSA clips ----------
urfd = FallCNNLSTM().to(device)
urfd.load_state_dict(torch.load('/content/drive/MyDrive/J26-DS-320/datasets/fall_cnn_lstm_v1.pt', map_location=device))
cross = metrics(y, predict(urfd, X))
print('\nURFD model -> GMDCSA (no retraining):', cross)

json.dump({"loso_per_subject": loso, "loso_pooled": pooled, "urfd_model_on_gmdcsa": cross,
           "n_clips": int(len(y))},
          open('/content/drive/MyDrive/J26-DS-320/datasets/gmdcsa24_results.json', 'w'), indent=2)

In [ ]:
  from google.colab import files
  files.download('/content/drive/MyDrive/J26-DS-320/datasets/gmdcsa24_results.json')